# EDA - EXPLORATORY DATA ANALYSIS

In [ ]:
import pandas as pd
from pathlib import Path


# load the own_sample

In [ ]:
"""Build ~100k rows without loading all of twcs.csv into memory (avoids OOM)."""
RAW_PATH = Path("../data/raw/twcs.csv")
OUT_DIR = Path("../data/own_sample")
OUT_PATH = OUT_DIR / "sample.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_ROWS_EST = 3_003_125
SAMPLE_N = 100_000
CHUNKSIZE = 150_000

parts = []
for i, chunk in enumerate(pd.read_csv(RAW_PATH, chunksize=CHUNKSIZE, low_memory=False)):
    take = max(1, int(SAMPLE_N * len(chunk) / TOTAL_ROWS_EST))
    take = min(take, len(chunk))
    parts.append(chunk.sample(n=take, random_state=42 + i))

parts_df = pd.concat(parts, ignore_index=True)
if len(parts_df) >= SAMPLE_N:
    df_sample = parts_df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
else:
    df_sample = parts_df.reset_index(drop=True)

df_sample.to_csv(OUT_PATH, index=False)
df = df_sample.copy()

print(f"Saved {len(df):,} rows -> {OUT_PATH}")


# check the RAG PART 

In [ ]:
rag_path = Path("../data/processed/rag_qa_pairs.csv")
if not rag_path.exists():
    raise FileNotFoundError(
        f"Missing {rag_path}. From project root run: python -m src.prepare_datasets --sample"
    )

rag_df = pd.read_csv(rag_path)
print(rag_df.shape)
print(rag_df.columns.tolist())
rag_df.head(10)


# UNDERSTAND THE STRUCTURE 

In [ ]:
df.shape
df.columns
df.head()

# CHECK THE DATA TYPE 

In [ ]:
df.info()

# NUMBER OF MISSING VALUES 

In [ ]:
df.isnull().sum()

# WHO IS SPEAKING ?

In [ ]:
df["inbound"].value_counts()

# AUTHOR ANALYSIS TO FILTER NOICE

In [ ]:
df["author_id"].value_counts().head(10)